# Example 1.4 / 3: Constraints Only Where They Apply

Companion notebook for **1.4 Constraints** in the AMPL lab Documentation, section "Constraints Only Where They Apply". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Constraints Only Where They Apply

A condition after a colon in the indexing expression skips some members. Here a stage whose
`avail` is 0 is treated as unlimited, so no constraint is generated for it.
`num_instances()` counts the constraints the declaration generated:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;
set STAGE;
param rate {PROD, STAGE} > 0;
param avail {STAGE} >= 0;
var Make {PROD} >= 0;

subject to Time {s in STAGE: avail[s] > 0}:
    sum {p in PROD} (1 / rate[p,s]) * Make[p] <= avail[s];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.set["STAGE"] = ["reheat", "roll"]
ampl.param["rate"] = {("bands", "reheat"): 200, ("bands", "roll"): 200,
                      ("coils", "reheat"): 200, ("coils", "roll"): 140}
ampl.param["avail"] = {"reheat": 0, "roll": 40}

In [5]:
# run.py
print(ampl.get_output("expand Time;"))
print(ampl.con["Time"].num_instances())

subject to Time['roll']:
	0.005*Make['bands'] + 0.00714286*Make['coils'] <= 40;


1


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils;
set STAGE := reheat roll;
param rate:   reheat  roll :=
  bands         200    200
  coils         200    140 ;
param avail := reheat 0  roll 40;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
expand Time;

subject to Time['roll']:
	0.005*Make['bands'] + 0.00714286*Make['coils'] <= 40;

